#### **Imported Libraries:**

In [36]:
import pygame
import random
from collections import deque


##### **Game Sprits:**

In [37]:
def load_images():
    images = {}
    images["green"] = pygame.image.load("assets/green_grass.png")
    images["light"] = pygame.image.load("assets/light_grass.png")
    images["white_chicken"] = pygame.image.load("assets/white_chicken.png")
    images["black_chicken"] = pygame.image.load("assets/black_chicken.png")
    images["white_egg"] = pygame.image.load("assets/white_egg.png")
    images["black_egg"] = pygame.image.load("assets/dark_egg.png")
    images["trap"] = pygame.image.load("assets/trap_hole.webp")
    images["turd"] = pygame.image.load("assets/chick_turd.webp")

    return images

##### **Game Screen Constants:**

In [38]:
WIDTH = 800
HEIGHT = 600

ROWS = 8
COLS = 8

TILE_SIZE = 64

In [39]:
MOVES = {
    pygame.K_UP: (-1, 0),
    pygame.K_DOWN: (1, 0),
    pygame.K_LEFT: (0, -1),
    pygame.K_RIGHT: (0, 1)
}

##### **Game Board Drawing Utility:**

In [40]:
def draw_board(screen, images):
    for row in range(ROWS):
        for col in range(COLS):
            x = col * TILE_SIZE
            y = row * TILE_SIZE

            if (row + col) % 2 == 0:
                tile = images["green"]
            else:
                tile = images["light"]

            tile = pygame.transform.scale(tile, (TILE_SIZE, TILE_SIZE))
            screen.blit(tile, (x, y))

In [41]:
def reachable_tiles(start, traps):
    visited = set()
    queue = deque([start])

    while queue:
        r, c = queue.popleft()

        if (r, c) in visited:
            continue
        if (r, c) in traps:
            continue

        visited.add((r, c))

        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < ROWS and 0 <= nc < COLS:
                queue.append((nr, nc))

    return visited

In [42]:
def generate_trapdoors(num_traps=12):
    import random

    forbidden = {
        (0, 0), (0, 1), (1, 0),
        (7, 7), (7, 6), (6, 7)
    }

    while True:
        traps = set()

        while len(traps) < num_traps:
            r = random.randint(0, ROWS - 1)
            c = random.randint(0, COLS - 1)

            if (r, c) not in forbidden:
                traps.add((r, c))

        white_reach = reachable_tiles((0, 0), traps)
        black_reach = reachable_tiles((7, 7), traps)

        MIN_ACCESSIBLE = 15  # tweak if needed

        if len(white_reach) >= MIN_ACCESSIBLE and len(black_reach) >= MIN_ACCESSIBLE:
            return traps

In [43]:
def draw_traps(screen, state, images):
    for (r, c) in state.revealed_traps:
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img = pygame.transform.scale(images["trap"], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [44]:
def draw_eggs(screen, state, images):
    for (r, c), player_id in state.eggs.items():
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img_key = "white_egg" if player_id == 0 else "black_egg"
        img = pygame.transform.scale(images[img_key], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [45]:
def draw_turds(screen, state, images):
    for (r, c) in state.turds:
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img = pygame.transform.scale(images["turd"], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [46]:
def draw_ui(screen, player1, player2, state):
    panel_x = COLS * TILE_SIZE
    panel_y = 0
    panel_w = 250
    panel_h = COLS * TILE_SIZE

    # Background color #25232c
    pygame.draw.rect(screen, (37, 35, 44), (panel_x, panel_y, panel_w, panel_h))

    # Font Setup
    title_font = pygame.font.SysFont('arial', 30, bold=True)
    subtitle_font = pygame.font.SysFont('arial', 14, bold=True)
    name_font = pygame.font.SysFont('arial', 20, bold=True)
    label_font = pygame.font.SysFont('arial', 12, bold=True)
    val_font = pygame.font.SysFont('arial', 24, bold=True)
    turn_font = pygame.font.SysFont('arial', 20, bold=True)

    # 1. Header
    screen.blit(title_font.render("Eggplosion", True, (255, 255, 255)), (panel_x + 20, 20))
    screen.blit(subtitle_font.render("MATCH STATS", True, (150, 150, 155)), (panel_x + 20, 65))

    # 2. Player Card Helper
    def draw_player_card(y_pos, name, is_white, player):
        # Lighter rounded rectangle
        card_rect = pygame.Rect(panel_x + 15, y_pos, 220, 145)
        pygame.draw.rect(screen, (50, 48, 60), card_rect, border_radius=12)

        # Color indicator circle
        circle_color = (255, 255, 255) if is_white else (20, 20, 20)
        pygame.draw.circle(screen, circle_color, (panel_x + 35, y_pos + 22), 8)
        if not is_white:
            pygame.draw.circle(screen, (100, 100, 100), (panel_x + 35, y_pos + 22), 8, 1)

        # Name
        screen.blit(name_font.render(name, True, (255, 255, 255)), (panel_x + 55, y_pos + 10))

        # Stats
        lbl_color = (150, 150, 155)
        
        # Row 1
        screen.blit(label_font.render("EGGS LAID", True, lbl_color), (panel_x + 30, y_pos + 45))
        screen.blit(val_font.render(str(player.eggs_placed), True, (255, 255, 255)), (panel_x + 30, y_pos + 60))
        
        screen.blit(label_font.render("TURDS LEFT", True, lbl_color), (panel_x + 130, y_pos + 45))
        screen.blit(val_font.render(str(player.turds_remaining), True, (255, 255, 255)), (panel_x + 130, y_pos + 60))

        # Row 2
        screen.blit(label_font.render("MOVES LEFT", True, lbl_color), (panel_x + 30, y_pos + 95))
        moves_color = (255, 255, 255) if player.moves_left > 10 else (255, 100, 100)
        screen.blit(val_font.render(str(player.moves_left), True, moves_color), (panel_x + 30, y_pos + 110))

    # Draw Cards
    draw_player_card(85, "White (Human):", True, player1)
    draw_player_card(250, "Black (Agent):", False, player2)

    # 3. Horizontal Line & Turn Indicator
    line_y = 415
    pygame.draw.line(screen, (80, 80, 90), (panel_x + 20, line_y), (panel_x + 230, line_y), 2)

    screen.blit(turn_font.render("Turn:", True, (180, 180, 185)), (panel_x + 20, line_y + 20))
    
    is_human = state.current_player_index == 0
    turn_text = "White" if is_human else "Black"
    turn_color = (255, 255, 255) if is_human else (180, 180, 180)
    
    turn_surface = turn_font.render(turn_text, True, turn_color)
    turn_x = panel_x + 230 - turn_surface.get_width()
    screen.blit(turn_surface, (turn_x, line_y + 20))

    # Action Phase text
    if state.awaiting_action and is_human:
        hint_font = pygame.font.SysFont('arial', 13, bold=True)
        screen.blit(hint_font.render("[E] Egg | [T] Turd", True, (100, 200, 100)), (panel_x + 20, line_y + 60))
 
def draw_dedication(screen):
    # Renders exactly below the board
    bar_y = COLS * TILE_SIZE
    bar_w = COLS * TILE_SIZE + 250
    bar_h = 50
    
    # Slightly darker than the UI panel for separation
    pygame.draw.rect(screen, (25, 25, 25), (0, bar_y, bar_w, bar_h))
    
    font = pygame.font.SysFont(None, 24, italic=True)
    text = font.render("This game is made in the honor of my twin sister Nour, to whom I will never be able to repay", True, (200, 200, 200))
    
    # Center text perfectly in the horizontal bar
    rect = text.get_rect(center=(bar_w // 2, bar_y + bar_h // 2))
    screen.blit(text, rect)

In [47]:
def draw_game_over(screen, status, winner_id, reason):
    overlay = pygame.Surface((762, 562))
    overlay.set_alpha(200)
    overlay.fill((10, 10, 15))
    screen.blit(overlay, (0, 0))
    
    if status == "WIN":
        if winner_id == 1:
            text1 = "Agent Wins!"
            color = (100, 255, 100) # Green
        else:
            text1 = "Agent Loses!"
            color = (255, 100, 100) # Red
    else:
        text1 = "It's a Tie!"
        color = (255, 255, 0) # Yellow
        
    large_font = pygame.font.SysFont('arial', 72, bold=True)
    small_font = pygame.font.SysFont('arial', 40)
    
    label1 = large_font.render(text1, True, color)

    
    r1 = label1.get_rect(center=(762 // 2, 562 // 2))

    
    screen.blit(label1, r1)


##### **Player Class and Functions:**

In [48]:
class Player:
    def __init__(self, row, col, image):
        self.row = row
        self.col = col
        self.image = image
        self.prev_row = row
        self.prev_col = col

        self.moves_left = 30
        self.turds_remaining = 4
        self.eggs_placed = 0

In [49]:
def draw_players(screen, players):
    for player in players:
        x = player.col * TILE_SIZE
        y = player.row * TILE_SIZE

        img = pygame.transform.scale(player.image, (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [50]:
def update_previous_position(player):
    player.prev_row = player.row
    player.prev_col = player.col

In [51]:
def get_current_player(state):
    return state.players[state.current_player_index]

In [52]:
def switch_turn(state):
    state.current_player_index = 1 - state.current_player_index

In [53]:
def move_player(state, direction):
    player = get_current_player(state)

    if player.moves_left <= 0:
        return "NO_MOVES"

    dr, dc = direction
    new_r = player.row + dr
    new_c = player.col + dc

    if not is_valid_move(state, player, new_r, new_c):
        return "ILLEGAL"

    # save previous
    update_previous_position(player)

    # move
    player.row = new_r
    player.col = new_c

    # resolve tile
    result = resolve_tile(state, player)

    if result == "TRAP":
        switch_turn(state)
        return "TRAP"

    return "SAFE"

In [54]:
def place_egg(state, player):
    pos = (player.row, player.col)

    if pos in state.eggs:
        return False
    if pos in state.turds:
        return False
    if pos in state.revealed_traps:
        return False

    state.eggs[pos] = state.current_player_index
    player.eggs_placed += 1
    
    # Play cluck sound for exactly 2000 milliseconds (2 seconds)
    if state.current_player_index == 0 and "cluck" in SOUNDS:
        SOUNDS["cluck"].play(maxtime=900)

    return True

In [55]:
def place_turd(state, player):
    pos = (player.row, player.col)

    if player.turds_remaining <= 0:
        return False
    if pos in state.eggs or pos in state.turds:
        return False

    state.turds.add(pos)
    player.turds_remaining -= 1
    
    # Play the turd sound
    if "turd" in SOUNDS:
        SOUNDS["turd"].play()

    return True

##### **Expectiminimax Algorithm:**

In [56]:
# Make the agent value eggs more early-game and turds more mid/late-game.
def game_progress(snap):
    total = 60  # 30 + 30 moves initially
    remaining = snap.moves_left[0] + snap.moves_left[1]
    return 1 - (remaining / total)  # 0 early → 1 late

In [57]:
class StateSnapshot:
    """
    A lightweight, pygame-free copy of GameState used by the search.
    All positions are (row, col) tuples.
    """
    def __init__(self, state):
        p0, p1 = state.players[0], state.players[1]
        self.pos          = [(p0.row, p0.col), (p1.row, p1.col)]
        self.moves_left   = [p0.moves_left, p1.moves_left]
        self.eggs_placed  = [p0.eggs_placed, p1.eggs_placed]
        self.turds_rem    = [p0.turds_remaining, p1.turds_remaining]
        self.eggs         = dict(state.eggs)          # {(r,c): player_id}
        self.turds        = set(state.turds)
        self.traps        = set(state.traps)          # all traps (hidden + revealed)
        self.revealed_traps = set(state.revealed_traps)
        self.current      = state.current_player_index

    def copy(self):
        s = StateSnapshot.__new__(StateSnapshot)
        s.pos           = list(self.pos)
        s.moves_left    = list(self.moves_left)
        s.eggs_placed   = list(self.eggs_placed)
        s.turds_rem     = list(self.turds_rem)
        s.eggs          = dict(self.eggs)
        s.turds         = set(self.turds)
        s.traps         = set(self.traps)
        s.revealed_traps = set(self.revealed_traps)
        s.current       = self.current
        return s

In [58]:
# Weights for H(s) = W1*(E_ai − E_human) + W2*(A_ai − A_human) + W3*(Tu_ai − Tu_human)
W1 = 10   # egg score difference (most important)
W2 = 1    # available moves difference
W3 = 2    # remaining turds difference (turds = board control)

AGENT_IDX = 1   # black chicken is player index 1
HUMAN_IDX = 0

def trap_probability(snap):
    """
    P_trap = (12 - T_found) / U
    where T_found = number of revealed traps, U = number of unknown (unrevealed) tiles.
    """
    total_tiles = ROWS * COLS
    revealed_count = len(snap.revealed_traps) + len(snap.turds) + len(snap.eggs)
    unknown = total_tiles - revealed_count
    t_found = len(snap.revealed_traps)
    remaining_traps = 12 - t_found
    if unknown <= 0 or remaining_traps <= 0:
        return 0.0
    return min(remaining_traps / unknown, 1.0)

def heuristic(snap):
    ai, hu = AGENT_IDX, HUMAN_IDX

    progress = game_progress(snap)

    # --- Egg importance increases early ---
    egg_weight = 12 + (1 - progress) * 9   # early ≈ 20, late ≈ 12

    # --- Turd importance increases later ---
    turd_weight = 2 + progress * 8.5        # early ≈ 2, late ≈ 12

    e_diff = snap.eggs_placed[ai] - snap.eggs_placed[hu]
    a_diff = snap.moves_left[ai] - snap.moves_left[hu]

    # Turd pressure near human
    turd_score = 0
    hr, hc = snap.pos[hu]
    for (tr, tc) in snap.turds:
        dist = abs(hr - tr) + abs(hc - tc)
        if dist <= 2:
            turd_score += (3 - dist) * turd_weight

    return egg_weight * e_diff + 1 * a_diff + turd_score

In [59]:
DIRS = [(-1,0),(1,0),(0,-1),(0,1)]

def get_legal_actions(snap, player_idx):
    """
    Returns a list of actions the agent can take from its current tile.
    Each action: ('egg', target_pos) or ('turd', target_pos)

    The agent walks from its current position (no BFS/DFS),
    checking adjacent tiles until it finds a safe empty one.
    Trap probability is handled at Chance nodes, not here.
    """
    r, c   = snap.pos[player_idx]
    opp    = snap.pos[1 - player_idx]
    actions = []

    # Tiles the agent can reach by stepping one tile in each direction
    candidates = set()
    queue = [(r, c)]
    visited = {(r, c)}

    # Simple flood: find all reachable tiles (no BFS depth limit,
    # but skip known-blocked tiles only)
    from collections import deque
    q = deque([(r, c)])
    while q:
        cr, cc = q.popleft()
        for dr, dc in DIRS:
            nr, nc = cr + dr, cc + dc
            if not (0 <= nr < ROWS and 0 <= nc < COLS): continue
            if (nr, nc) in visited: continue
            if (nr, nc) == opp: continue
            if (nr, nc) in snap.turds: continue
            if (nr, nc) in snap.revealed_traps: continue
            visited.add((nr, nc))
            candidates.add((nr, nc))
            q.append((nr, nc))

    for pos in candidates:
        # Egg: tile must be empty (no egg, no turd, not a revealed trap)
        if pos not in snap.eggs and pos not in snap.turds and pos not in snap.revealed_traps:
            actions.append(('egg', pos))
        # Turd: tile must be empty, player must have turds left
        if snap.turds_rem[player_idx] > 0 and pos not in snap.eggs and pos not in snap.turds:
            actions.append(('turd', pos))

    return actions

In [60]:
def apply_action(snap, player_idx, action):
    """Returns a new StateSnapshot after applying action. Does NOT modify snap."""
    s = snap.copy()
    kind, pos = action

    if kind == 'egg':
        s.eggs[pos] = player_idx
        s.eggs_placed[player_idx] += 1
        s.moves_left[player_idx]  -= 1

    elif kind == 'turd':
        s.turds.add(pos)
        s.turds_rem[player_idx]  -= 1
        s.moves_left[player_idx] -= 1

    s.current = 1 - player_idx
    return s

In [61]:
SEARCH_DEPTH = 3   # keep low — algorithm is expensive

def expectiminimax(snap, depth, node_type):
    """
    node_type: 'max' | 'min' | 'chance'
    Returns a numeric score from the agent's perspective.
    """
    # Terminal / depth limit
    if depth == 0 or snap.moves_left[AGENT_IDX] <= 0 or snap.moves_left[HUMAN_IDX] <= 0:
        return heuristic(snap)

    if node_type == 'max':
        actions = get_legal_actions(snap, AGENT_IDX)
        if not actions:
            return heuristic(snap)
        best = float('-inf')
        for action in actions:
            child = apply_action(snap, AGENT_IDX, action)
            # After agent acts → chance node (will the human step on a trap?)
            val = expectiminimax(child, depth - 1, 'chance')
            if val > best:
                best = val
        return best

    elif node_type == 'min':
        actions = get_legal_actions(snap, HUMAN_IDX)
        if not actions:
            return heuristic(snap)
        worst = float('inf')
        for action in actions:
            child = apply_action(snap, HUMAN_IDX, action)
            val = expectiminimax(child, depth - 1, 'max')
            if val < worst:
                worst = val
        return worst

    else:  # chance
        p_trap = trap_probability(snap)
        p_safe = 1.0 - p_trap

        # --- Trap outcome ---
        trap_snap = snap.copy()
        trap_snap.moves_left[trap_snap.current] -= 2
        trap_snap.current = 1 - trap_snap.current   # ✅ turn ends

        trap_val = heuristic(trap_snap)

        # --- Safe outcome ---
        safe_snap = snap.copy()
        safe_snap.current = 1 - safe_snap.current   # after action, opponent plays
        safe_val = expectiminimax(safe_snap, depth - 1, 'min')

        return p_trap * trap_val + p_safe * safe_val

In [62]:
def agent_turn(state):
    """
    Called once per agent turn.
    """
    snap = StateSnapshot(state)
    actions = get_legal_actions(snap, AGENT_IDX)

    if not actions:
        # No moves available — force turn switch
        state.players[AGENT_IDX].moves_left -= 1
        switch_turn(state)
        return

    # Score each action with expectiminimax
    best_action = None
    best_score  = float('-inf')
    for action in actions:
        child = apply_action(snap, AGENT_IDX, action)

        # 🔥 NEW: penalize far targets
        (tr, tc) = action[1]
        ar, ac = snap.pos[AGENT_IDX]
        distance = abs(tr - ar) + abs(tc - ac)

        distance_penalty = -2 * distance   # tune this (1.0 → 2.5)

        score = expectiminimax(child, SEARCH_DEPTH - 1, 'chance') + distance_penalty

        if score > best_score:
            best_score = score
            best_action = action

    kind, (tr, tc) = best_action
    agent  = state.players[AGENT_IDX]

    # --- Walk the agent to the target tile, one step at a time ---
    max_steps = ROWS + COLS   
    steps = 0
    
    # FIX 1: Keep track of visited tiles during the walk to prevent infinite bouncing
    visited_walk = {(agent.row, agent.col)}

    while (agent.row, agent.col) != (tr, tc) and steps < max_steps:
        moved = False
        dr = tr - agent.row
        dc = tc - agent.col
        
        preferred = []
        if dr != 0: preferred.append((1 if dr > 0 else -1, 0))
        if dc != 0: preferred.append((0, 1 if dc > 0 else -1))

        # Build fallback directions safely
        fallback = [d for d in DIRS if d not in preferred]

        for ddr, ddc in preferred + fallback:
            nr, nc = agent.row + ddr, agent.col + ddc
            
            # Now it checks visited_walk so it is forced to move THROUGH the egg to an empty tile
            if is_valid_move(state, agent, nr, nc) and (nr, nc) not in visited_walk:
                update_previous_position(agent)
                agent.row, agent.col = nr, nc
                visited_walk.add((nr, nc)) # Log the step
                
                result = resolve_tile(state, agent)
                if result == "TRAP":
                    switch_turn(state)
                    return
                moved = True
                break

        if not moved:
            break  # completely stuck, give up

        steps += 1

    # --- Arrived at (or near) target — check tile is still usable ---
    pos = (agent.row, agent.col)

    if kind == 'egg' and pos not in state.eggs and pos not in state.turds and pos not in state.revealed_traps:
        place_egg(state, agent)
    elif kind == 'turd' and agent.turds_remaining > 0 and pos not in state.eggs and pos not in state.turds:
        place_turd(state, agent)
    else:
        # Target tile became occupied — move to any safe adjacent tile and try again
        for ddr, ddc in DIRS:
            nr, nc = agent.row + ddr, agent.col + ddc
            if is_valid_move(state, agent, nr, nc) and (nr, nc) not in state.eggs and (nr, nc) not in state.turds:
                update_previous_position(agent)
                agent.row, agent.col = nr, nc
                result = resolve_tile(state, agent)
                if result == "TRAP":
                    switch_turn(state)
                    return
                if kind == 'egg':
                    place_egg(state, agent)
                else:
                    place_turd(state, agent)
                break

    agent.moves_left -= 1
    switch_turn(state)

##### **Game Logic Utility:**

In [63]:
def show_main_menu(screen, bg_image):
    # Play the rooster sound when the menu opens
    if "rooster" in SOUNDS:
        SOUNDS["rooster"].play()

    title_font = pygame.font.SysFont('arial', 80, bold=True)
    button_font = pygame.font.SysFont('arial', 40, bold=True)
    
    # Define button dimensions and position (Centered)
    button_w, button_h = 200, 60
    button_x = (WIDTH // 2) - (button_w // 2)
    button_y = (HEIGHT // 2) + 50
    button_rect = pygame.Rect(button_x, button_y, button_w, button_h)

    menu_running = True
    while menu_running:
        screen.blit(bg_image, (0, 0))
        
        # Get mouse position to check for hover effects
        mouse_pos = pygame.mouse.get_pos()
        is_hovering = button_rect.collidepoint(mouse_pos)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                exit() # Exit the program completely
            
            if event.type == pygame.MOUSEBUTTONDOWN:
                if event.button == 1 and is_hovering:
                    # Clicked the start button!
                    return # Exit the menu loop and proceed to the game

        # --- Drawing ---
        # 1. Draw Title Text with a slight drop shadow for readability
        title_text = title_font.render("EGGPLOSION", True, (255, 255, 255))
        shadow_text = title_font.render("EGGPLOSION", True, (30, 30, 30))
        text_rect = title_text.get_rect(left=50, centery=HEIGHT // 2 - 50)
        
        screen.blit(shadow_text, (text_rect.x + 4, text_rect.y + 4))
        screen.blit(title_text, text_rect)

        # 2. Draw Start Button (Change color if hovering)
        button_color = (100, 200, 100) if is_hovering else (70, 160, 70)
        pygame.draw.rect(screen, button_color, button_rect, border_radius=15)
        pygame.draw.rect(screen, (40, 100, 40), button_rect, width=4, border_radius=15) # Border
        
        btn_text = button_font.render("START", True, (255, 255, 255))
        button_rect.left = 50
        btn_text_rect = btn_text.get_rect(center=button_rect.center)
        screen.blit(btn_text, btn_text_rect)

        pygame.display.flip()
        clock.tick(60)

In [64]:
def handle_action_phase(state, player, event):
    if event.key == pygame.K_e:
        success = place_egg(state, player)

    elif event.key == pygame.K_t:
        success = place_turd(state, player)

    else:
        return  # wait for valid action key

    if success:
        player.moves_left -= 1
        switch_turn(state)

In [65]:
def is_valid_move(state, player, new_r, new_c):
    # outside board
    if not (0 <= new_r < ROWS and 0 <= new_c < COLS):
        return False

    # opponent position
    opponent = state.players[1 - state.current_player_index]
    if (new_r, new_c) == (opponent.row, opponent.col):
        return False

    # turd
    if (new_r, new_c) in state.turds:
        return False

    # revealed trap
    if (new_r, new_c) in state.revealed_traps:
        return False

    return True

In [66]:
def handle_input(state, event):
    # Ignore input during agent turn
    if state.current_player_index == AGENT_IDX:
        return

    if event.key in MOVES:
        result = move_player(state, MOVES[event.key])

        if result == "ILLEGAL":
            return

        if result == "TRAP":
            state.awaiting_action = False 
            return

        # SAFE → go to action phase
        state.awaiting_action = True
        return

    # Action phase
    if state.awaiting_action:
        player = get_current_player(state)
        handle_action_phase(state, player, event)
        state.awaiting_action = False

In [67]:
def resolve_tile(state, player):
    pos = (player.row, player.col)

    # Trapdoor
    if pos in state.traps:
        state.revealed_traps.add(pos)

        # penalty
        player.moves_left -= 2

        # return back
        player.row = player.prev_row
        player.col = player.prev_col
        
        # Play the trapdoor hit sound
        if "trap" in SOUNDS:
            SOUNDS["trap"].play()

        return "TRAP"

    return "SAFE"

In [68]:
class GameState:
    def __init__(self, player1, player2, traps):
        self.players = [player1, player2]
        self.current_player_index = 0

        self.traps = traps
        self.revealed_traps = set()

        self.turds = set()
        self.eggs = {}  # (row, col) → player_id

        self.awaiting_action = False

In [69]:
def has_available_moves(state, player):
    """True if the player can reach at least one tile from their position."""
    from collections import deque
    start = (player.row, player.col)
    visited = {start}
    queue = deque([start])
    player_idx = state.players.index(player)
    opponent = state.players[1 - player_idx]
    opp_pos = (opponent.row, opponent.col)

    while queue:
        r, c = queue.popleft()
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < ROWS and 0 <= nc < COLS): continue
            if (nr, nc) in visited: continue
            if (nr, nc) == opp_pos: continue
            if (nr, nc) in state.turds: continue
            if (nr, nc) in state.revealed_traps: continue
            visited.add((nr, nc))
            queue.append((nr, nc))

    # More than 1 means the player can reach tiles beyond their own square
    return len(visited) > 1

def check_game_over(state):
    p1, p2 = state.players[0], state.players[1]
    
    # 1. Blocking/Trapping Condition
    if not state.awaiting_action:
        current_player = get_current_player(state)
        if not has_available_moves(state, current_player):
            winner_idx = 1 - state.current_player_index
            return "WIN", winner_idx, "Opponent Trapped!"

    # 2. Complete Exhaustion Condition (Both out of moves)
    if p1.moves_left <= 0 and p2.moves_left <= 0:
        if p1.eggs_placed > p2.eggs_placed:
            return "WIN", 0, "Most Eggs Laid!" # Changed from "Least"
        elif p2.eggs_placed > p1.eggs_placed:
            return "WIN", 1, "Most Eggs Laid!" # Changed from "Least"
        else:
            return "TIE", None, "Tie: Equal Eggs!"
            
    # 3. Sudden Death (One player is out of moves)
    if p1.moves_left <= 0 and p2.moves_left > 1:
        return "WIN", 1, "Opponent Out of Moves!" # Fixed message
    if p2.moves_left <= 0 and p1.moves_left > 1:
        return "WIN", 0, "Opponent Out of Moves!" # Fixed message
        
    return "PLAYING", None, None


In [70]:
pygame.init()
pygame.mixer.init()

# --- Exact Screen Dimensions ---
BOARD_SIZE = COLS * TILE_SIZE # 512
PANEL_WIDTH = 250
BOTTOM_HEIGHT = 50

WIDTH = BOARD_SIZE + PANEL_WIDTH  # 762
HEIGHT = BOARD_SIZE + BOTTOM_HEIGHT # 562

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Eggplosion")

# Load Sounds
SOUNDS = {}

SOUNDS["cluck"] = pygame.mixer.Sound("assets/Chicken_clucking_sound.mp3")
SOUNDS["turd"] = pygame.mixer.Sound("assets/Turd_sound.mp3")
SOUNDS["trap"] = pygame.mixer.Sound("assets/Trapdoor_hit_sound.mp3")
SOUNDS["agent_win"] = pygame.mixer.Sound("assets/Win_sound.mp3")
SOUNDS["agent_lose"] = pygame.mixer.Sound("assets/Lose_sound.mp3")
SOUNDS["tie"] = pygame.mixer.Sound("assets/Tie_sound.mp3")
SOUNDS["rooster"] = pygame.mixer.Sound("assets/rooster.mp3")


images = load_images()
menu_bg = pygame.image.load("assets/menu.png").convert()
menu_bg = pygame.transform.scale(menu_bg, (WIDTH, HEIGHT))

player1 = Player(0, 0, images["white_chicken"])
player2 = Player(7, 7, images["black_chicken"])

traps = generate_trapdoors()
state = GameState(player1, player2, traps)

running = True
clock = pygame.time.Clock()

game_status = "PLAYING"
winner_id = None
win_reason = ""
end_sound_played = False

show_main_menu(screen, menu_bg)

while running:
    clock.tick(60)
    
    # Check Win/Tie Conditions
    if game_status == "PLAYING":
        status, w_id, reason = check_game_over(state)
        if status != "PLAYING":
            game_status = status
            winner_id = w_id
            win_reason = reason

    # Play End Game Sound Once
    if game_status != "PLAYING" and not end_sound_played:
        end_sound_played = True
        if game_status == "WIN":
            if winner_id == 1 and "agent_win" in SOUNDS:
                SOUNDS["agent_win"].play()
            elif winner_id == 0 and "agent_lose" in SOUNDS:
                SOUNDS["agent_lose"].play()
        elif game_status == "TIE" and "tie" in SOUNDS:
            SOUNDS["tie"].play()

    # Handle Events
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN and game_status == "PLAYING":
            handle_input(state, event)

    # Process Turns
    if game_status == "PLAYING":
        current_p = state.players[state.current_player_index]
        if current_p.moves_left <= 0:
            switch_turn(state)
        elif state.current_player_index == AGENT_IDX and not state.awaiting_action:
            
            # --- FORCE SCREEN UPDATE ---
            # This makes sure the UI visibly changes to "Turn: Black" 
            # while the agent is frozen calculating its next move.
            screen.fill((0, 0, 0))
            draw_board(screen, images)
            draw_traps(screen, state, images)
            draw_eggs(screen, state, images)
            draw_turds(screen, state, images)
            draw_players(screen, state.players)
            draw_ui(screen, player1, player2, state)
            draw_dedication(screen)
            pygame.display.flip()
            # ---------------------------
            
            agent_turn(state)

    # Drawing
    screen.fill((0, 0, 0)) # Fill black to cover any rounding
    draw_board(screen, images)
    draw_traps(screen, state, images)
    draw_eggs(screen, state, images)
    draw_turds(screen, state, images)
    draw_players(screen, state.players)
    
    draw_ui(screen, player1, player2, state)
    draw_dedication(screen)
    
    if game_status != "PLAYING":
        draw_game_over(screen, game_status, winner_id, win_reason)
        
    pygame.display.flip()

pygame.quit()


error: Library not initialized

: 